In [2]:
import numpy as np

np.random.seed(42)

X = np.random.uniform(-2, 2, (400, 3))
y = (
    np.sin(X[:, 0]) +
    0.5 * (X[:, 1] ** 2) -
    0.8 * X[:, 2]
).reshape(-1, 1)

def relu(z): return np.maximum(0, z)
def drelu(z): return (z > 0).astype(float)

def sigmoid(z): return 1 / (1 + np.exp(-z))
def dsigmoid(z):
    s = sigmoid(z)
    return s * (1 - s)

def init_params(layer_dims):
    params = {}
    for l in range(1, len(layer_dims)):
        params["W"+str(l)] = np.random.uniform(-0.5,0.5,(layer_dims[l],layer_dims[l-1]))
        params["b"+str(l)] = np.zeros((layer_dims[l],1))
    return params

def forward(X, params, activation):
    A = X.T
    caches = []
    L = len(params)//2
    for l in range(1, L):
        W = params["W"+str(l)]
        b = params["b"+str(l)]
        Z = W @ A + b
        A = activation(Z)
        caches.append((A,Z))
    W = params["W"+str(L)]
    b = params["b"+str(L)]
    Z = W @ A + b
    caches.append((A,Z))
    return Z, caches

def mse(y, y_hat):
    return np.mean((y_hat - y.T)**2)

def backward(X, y, params, caches, activation, dactivation):
    grads = {}
    L = len(params)//2
    m = X.shape[0]
    y_hat,_ = forward(X,params,activation)
    dZ = 2*(y_hat - y.T)/m
    for l in reversed(range(1,L+1)):
        A_prev = X.T if l==1 else caches[l-2][0]
        W = params["W"+str(l)]
        grads["dW"+str(l)] = dZ @ A_prev.T
        grads["db"+str(l)] = np.sum(dZ,axis=1,keepdims=True)
        if l>1:
            dA_prev = W.T @ dZ
            Z_prev = caches[l-2][1]
            dZ = dA_prev * dactivation(Z_prev)
    return grads

def update(params, grads, lr):
    for l in range(1,len(params)//2+1):
        params["W"+str(l)] -= lr*grads["dW"+str(l)]
        params["b"+str(l)] -= lr*grads["db"+str(l)]
    return params

def train(layer_dims, activation, dactivation):
    params = init_params(layer_dims)
    lr = 0.01
    epochs = 1000
    for epoch in range(epochs):
        y_hat,caches = forward(X,params,activation)
        loss = mse(y,y_hat)
        grads = backward(X,y,params,caches,activation,dactivation)
        params = update(params,grads,lr)
        if epoch==199:
            loss200 = loss
    grad_l1 = np.linalg.norm(grads["dW1"])
    grad_last = np.linalg.norm(grads["dW"+str(len(layer_dims)-1)])
    return loss,loss200,grad_l1,grad_last

models = {
    "A":[3,4,1],
    "B":[3,6,6,1],
    "C":[3,8,8,8,8,1],
    "D":[3,8,8,8,8,8,8,8,8,1]
}

activations = {
    "ReLU":(relu,drelu),
    "Sigmoid":(sigmoid,dsigmoid)
}

for m_name, dims in models.items():
    for a_name,(act,dact) in activations.items():
        final_loss,loss200,g1,gL = train(dims,act,dact)
        print("Model:",m_name,"Activation:",a_name)
        print("Loss@200:",loss200)
        print("Final Loss:",final_loss)
        print("Grad Norm L1:",g1)
        print("Grad Norm Last:",gL)
        print("-----")

Model: A Activation: ReLU
Loss@200: 0.4938471859377117
Final Loss: 0.11154512827725767
Grad Norm L1: 0.04521736074874136
Grad Norm Last: 0.03727266746906128
-----
Model: A Activation: Sigmoid
Loss@200: 1.370388928827891
Final Loss: 0.4158099572141786
Grad Norm L1: 0.03787434410546774
Grad Norm Last: 0.06391580933406224
-----
Model: B Activation: ReLU
Loss@200: 1.3210489939446595
Final Loss: 0.09423018695250945
Grad Norm L1: 0.05546783243752947
Grad Norm Last: 0.041026395838443176
-----
Model: B Activation: Sigmoid
Loss@200: 1.6998648936650078
Final Loss: 0.9997871979670778
Grad Norm L1: 0.22817438016634625
Grad Norm Last: 0.26779745129203136
-----
Model: C Activation: ReLU
Loss@200: 0.5084766327011095
Final Loss: 0.057146199947810006
Grad Norm L1: 0.028670585360300687
Grad Norm Last: 0.004873098842823028
-----
Model: C Activation: Sigmoid
Loss@200: 1.7419838103342904
Final Loss: 1.7409059177368322
Grad Norm L1: 0.004393241186548978
Grad Norm Last: 0.007335667782914393
-----
Model: D Ac

Model A: 3 → 4 → 1

Layer 1:
4 × 3 + 4 = 16

Output Layer:
1 × 4 + 1 = 5

Total Parameters = 16 + 5 = 21

Model B: 3 → 6 → 6 → 1

Layer 1:
6 × 3 + 6 = 24

Layer 2:
6 × 6 + 6 = 42

Output Layer:
1 × 6 + 1 = 7

Total Parameters = 24 + 42 + 7 = 73

Model C: 3 → 8 → 8 → 8 → 8 → 1

Layer 1:
8 × 3 + 8 = 32

Layer 2:
8 × 8 + 8 = 72

Layer 3:
8 × 8 + 8 = 72

Layer 4:
8 × 8 + 8 = 72

Output Layer:
1 × 8 + 1 = 9

Total Parameters = 32 + 72 + 72 + 72 + 9 = 257

Model D: 3 → 8 → 8 → 8 → 8 → 8 → 8 → 8 → 8 → 1

Layer 1:
8 × 3 + 8 = 32

Hidden layers (7 layers of 8→8):
7 × (8 × 8 + 8) = 7 × 72 = 504

Output Layer:
1 × 8 + 1 = 9

Total Parameters = 32 + 504 + 9 = 545